In [0]:
%pip install xgboost
%restart_python

In [0]:
import importlib
import pandas as pd
import numpy as np
import src.utils.fetch_lighthouse_data as fld
importlib.reload(fld)
from scipy.stats import percentileofscore
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from xgboost import XGBRegressor

API_KEY = dbutils.secrets.get(
    scope="site_speed_project",
    key="google_psi_api_key"
)

In [0]:
# result = fld.fetch_data(api_key=API_KEY, url="https://www.hafuboti.com/")

In [0]:
df = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score IS NOT NULL
''').toPandas()

In [0]:
df_clipped = df.dropna().copy()
numeric_cols = df_clipped.select_dtypes(include="number").columns

for col in numeric_cols:
    lower = df_clipped[col].quantile(0.00)
    upper = df_clipped[col].quantile(0.99)
    df_clipped[col] = df_clipped[col].clip(lower, upper)

In [0]:
def print_stats(df, col):
    fig, ax = plt.subplots(2, 1, figsize=(4,4))
    ax[0].hist(df[col], bins=20)
    ax[1] = df[col].plot(kind="box", logx=True, vert=False)

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print("lower: ", lower)
    print("upper: ", upper)
    print(df[col].describe(percentiles=[0.9,0.95,0.99,0.995,0.999]))


In [0]:
df_clipped.columns

In [0]:
df_clipped.shape

In [0]:
drop_cols = [
    'url',
    'strategy',
    'domain',
    "performance_score",
    'speed-index',
    'largest-contentful-paint',
    'cumulative-layout-shift',
    'first-contentful-paint',
    'total-blocking-time',
    'EXPERIMENTAL_TIME_TO_FIRST_BYTE',
    'INTERACTION_TO_NEXT_PAINT'
]
X = df_clipped.drop(columns=drop_cols)
y = df_clipped["largest-contentful-paint"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
X.columns

In [0]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")
print(f"Mean Squared Error: {mse}")
# importance = model.coef_
# print(f"Feature Importance: {importance}")

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso_model = LassoCV(cv=5)
lasso_model.fit(X_train_scaled, y_train)
print(lasso_model.score(X_train_scaled, y_train))
y_pred = lasso_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}") 
print(f"Mean Squared Error: {mse}")


In [0]:
pd.DataFrame(lasso_model.coef_, list(X_train.columns))

In [0]:
en_model = ElasticNetCV(cv=5, l1_ratio=0.5, random_state=42)
en_model.fit(X_train_scaled, y_train)
print(en_model.score(X_train_scaled, y_train))

y_pred = en_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")
print(f"Mean Squared Error: {mse}")


In [0]:
pd.DataFrame(en_model.coef_, list(X_train.columns))

In [0]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
print(rf_model.score(X_train, y_train))

y_pred = rf_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")
print(f"Mean Squared Error: {mse}")


In [0]:
gbr = GradientBoostingRegressor(random_state=42)
print(cross_val_score(gbr, X_train, y_train, cv=5))

gbr.fit(X_train, y_train)
print(gbr.score(X_train, y_train))
y_pred = gbr.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")
print(f"Mean Squared Error: {mse}")

In [0]:
gbr.feature_importances_
importances = pd.DataFrame({'feature':X_train.columns,'importance': np.round(gbr.feature_importances_,3)})
importances = importances.sort_values('importance',ascending=False).set_index('feature')
top_features = importances.head(10).index
importances.head(20)

In [0]:
gbr2 = GradientBoostingRegressor(random_state=42)
scores = cross_val_score(gbr2, X_train[top_features], y_train, cv=5, scoring="neg_mean_absolute_error")
print("Scores AVG: ", scores.mean())
print("Scores STD: ", scores.std())

gbr2.fit(X_train[top_features], y_train)
print(gbr2.score(X_train[top_features], y_train))
y_pred = gbr2.predict(X_test[top_features])

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
# # grid_param = {
# #     "n_estimators": [50, 100, 300, 500],
# #     "learning_rate": [0.0001, 0.001, 0.01, 0.1],
# #     "max_depth": [3, 5, 7],
# #     "subsample": [0.5, 0.7, 1.0]
# # }
# param_grid = {
#     "n_estimators": [50, 100, 200, 300, 500],
#     "learning_rate": [0.01, 0.03, 0.05, 0.1],
#     "max_depth": [2, 3, 4, 5],
#     "subsample": [0.7, 0.8, 0.9, 1.0],
#     "min_samples_leaf": [1, 5, 10, 20]
# }
# grid_search = GridSearchCV(gbr2, param_grid, cv=5, scoring="neg_mean_absolute_error", verbose=1, n_jobs=-1)
# grid_search.fit(X_train[top_features], y_train)
# print(grid_search.best_params_)

In [0]:
# {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 500, 'subsample': 1.0}

gbr2 = GradientBoostingRegressor(learning_rate=0.1, max_depth=7, n_estimators=500, subsample=1.0, random_state=42)
scores = cross_val_score(gbr2, X_train[top_features], y_train, cv=5, scoring="neg_mean_absolute_error")
print("Scores AVG: ", scores.mean())
print("Scores STD: ", scores.std())

gbr2.fit(X_train[top_features], y_train)
print(gbr2.score(X_train[top_features], y_train))
y_pred = gbr2.predict(X_test[top_features])

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
# {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 500, 'subsample': 0.8}

gbr3 = GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=500, subsample=0.8, min_samples_leaf=5, random_state=42)
scores = cross_val_score(gbr3, X_train[top_features], y_train, cv=5, scoring="neg_mean_absolute_error")
print("Scores AVG: ", scores.mean())
print("Scores STD: ", scores.std())

gbr3.fit(X_train[top_features], y_train)
print(gbr3.score(X_train[top_features], y_train))
y_pred = gbr3.predict(X_test[top_features])

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
%skip
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror"
)

scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print("XGBoost CV scores:", scores)
print("Mean CV R2:", scores.mean())
print("CV std:", scores.std())